# Gram Newton Schulz
## The Goal
Let $G \in \R^{m \times n}$ where $m \leq n = \alpha m$. We want to compute
$$\mathrm{polar}(G) = (GG^\top)^{-1/2} G$$

The standard Newton-Schulz-like iteration is $$X_0 = G, \quad X_t = p_t(X_{t-1})$$ where $p_t$ is a polynomial like $p_t(M) = \tfrac32 M - \tfrac12 MM^\top M$. Here, $(p_1, p_2, \ldots)$ is a sequence of odd polynomials satisfying $$\lim_{T \to \infty} (p_T \circ \cdots \circ p_1)(x) \to 1 \quad \forall x \in (0, 1]$$
This ensures that Newton Schulz converges for all $G$ such that $\|G\|_2 \leq 1$.
If run for $T$ iterations, the runtime of Newton Schulz is $\approx 2\alpha T m^3$.

The alternative version aims to do as much computation as possible with small $m \times m$ matrices. We call this method "Gram Newton Schulz". The outline of the method is as follows:
1. Form the Gram matrix $GG^\top$
2. Compute the approximation $Q_T \approx (GG^\top)^{-1/2}$ using an iterative polynomial method
3. Form $Q_T G$

Steps 1 and 3 each require multiplying $m \times n$ matrices. However, step 2 does not. Therefore, the runtime of Gram Newton Schulz is just $(3T + 2\alpha)m^3$. Figuring $T=5, \alpha=4$, this version is 43\% faster. (When we instead use degree-5 polynomials $p_t(x) = ax + bx^3 + cx^5$, standard Newton Schulz costs $(2\alpha + 1)Tm^3$ while Gram Newton Schulz costs $(4T + 2\alpha)m^3$, 38\% faster.)

## Derivation from Standard Newton Schulz
It remains to construct an approximation to $(GG^\top)^{-1/2}$. As wtih Newton Schulz, we restrict ourselves to methods based on matrix polynomials.
As it turns out, such a method can be derived from any sequence of odd polynomials $(p_1, p_2, \ldots)$ that satisfies the convergence criterion for Newton Schulz given above. If we use this 
If we use this method in step 2, then the overall Gram Newton Schulz procedure will be exactly equivalent to standard Newton Schulz in exact arithmetic.

We begin with the scalar analogue of the method.
Let $x_0 \in (0, 1]$ and $x_t = p_t(x_{t-1})$, as in standard Newton Schulz.
Since $p_t$ is odd, it can be written as $p_t(x) = x h_t(x^2)$, where $h_t$ is a polynomial like $h_t(x) = \tfrac32 - \tfrac12 x$.
Define 
Define $r_t := x_t^2$ and $z_t := h_t(r_{t-1})$. Then
$$x_{t} = p_t(x_{t-1}) = x_{t-1} h_t(x_{t-1}^2) = x_{t-1} h_t(r_{t-1}) = x_{t-1} z_t$$
Squaring both sides yields
$$r_t = r_{t-1} z_t^2$$
If we define $q_t := x_t / x_0$, then we can also derive
$$q_t = q_{t-1} z_t$$
Finally, $1 = \lim_{T \to \infty} x_T = \lim_{T \to \infty} q_T x_0 = \lim_{T \to \infty} q_T \sqrt{r_0} \implies q_T \to 1/\sqrt{r_0}$. Thus, the following iterative method computes the inverse square root of $r_0$:
- $z_t = h_t(r_{t-1})$
- $r_t = r_{t-1} z_t^2$
- $q_t = q_{t-1} z_t$&emsp;(where we initialize $q_0 = 1$).

Furthermore, if we initialize this method with $r_0 = x_0^2$, then our analysis shows that $x_T = q_T x_0$ is precisely the output of standard Newton Schulz. Thus, we have computed $x_T$ from $x_0$ without ever constructing the intermediate values $x_1, \ldots X_{T-1}$.

To obtain Gram Newton Schulz, we simply lift the above procedure to matrices.
As in standard Newton Schulz, each operation preserves eigenvectors/singular vectors.
Therefore, each eigenvalue / singular value evolves independently of the others according to the scalar iteration described above.

> ### Gram Newton Schulz (Version 1 - High Precision)
> Input: $G \in \R^{m \times n}$ with $m \leq n$ and $\|G\|_2 \leq 1$.
>
> Initialize: $R_0 = GG^\top$ and $Q_0 = I$.
>
> Repeat for $t=1, \ldots,\,T$:
> - $Z_t = h_t(R_{t-1})$&emsp;&emsp;(e.g. $\tfrac32 I - \tfrac12 R_{t-1}$)
> - $R_t = Z_t^\top R_{t-1} Z_t$
> - $Q_t = Q_{t-1}Z_t$
>
> Output: $X_T = Q_T G$.

This method is almost the same as Polar Express, Appendix F.

## Tracking the Spectra in Gram Newton Schulz 

We will now demonstrate Gram Newton Schulz on a simple synthetic example of a $128 \times 512$ matrix with an exponentially decaying spectrum. We will use the degree-5 Newton Schulz polynomial $p_t(x) = \tfrac{15}8 x - \tfrac{10}8 x^3 + \tfrac38 x^5$.

By construction, $Z_t, R_t$, and $Q_t$ are symmetric.
Because the only operations we perform are matrix polynomials, the eigenvectors of all of these matrices are identical, and they match the left singular vectors of $X_t$.
We can therefore plot the eigenvalues of $R_t$ and $Q_t$ against the corresponding singular values of $X_0$ to trace how each evolves according to the polynomial iterations.
Even though our method does not need to compute the intermediate matrices $X_1, \ldots X_{T-1}$, we do so here for demonstration.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.scale import AsinhScale
import numpy as np
import pandas as pd
import torch

%load_ext autoreload
%autoreload 2
from appF_diagnostic import *

DEVICE = 'cuda:1'

In [ ]:
n = 128
aspect_ratio = 4
spectrum = torch.logspace(-0.01, -8, steps=n, dtype=torch.float64, device=DEVICE)
G = spectrum2matrix(spectrum, aspect_ratio)

_, f64_diagnostics = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=10,
    ambient_dtype=torch.float64,
)(G)

HTML(spectrum_evolution_plot(f64_diagnostics).to_jshtml())

Initially, we have $r_0 = x_0^2$, and $q_0 = 1$. As training procedes, $q_t$ approaches $1/\sqrt{r_0} = 1/x_0$, and so $x_t = q_t x_0$ approaches $1$. Since $r_t = x_t^2$, it too approaches 1. (We can also see the gap between $r_t$ and $1$ as a measure of the error in the approximation $q_t \approx 1/\sqrt{r_0}$, since $r_t = q_t^2 r_0$.) Note that if $x_0$ is close to 1, the method converges quickly, while if $x_0$ is close to zero, it converges slowly. After 10 iterations, the spectrum of $X_t$ is visually indistinguishable from $1$.

So far, this is all expected. As we showed above, Gram Newton Schulz is exactly equivalent to Newton Schulz in exact arithmetic.

## Trouble in Low Precision
Unfortunately, in floating point arithemtic, Gram Newton Schulz is not equivalent to the standard version; it is numerically unstable. Let's see what happens when we repeat the previous experiment in `bfloat16` arithmetic.

In [ ]:
_, f16_diagnostics = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=10,
    ambient_dtype=torch.bfloat16
)(G)

HTML(spectrum_evolution_plot(f16_diagnostics).to_jshtml())

The first few iterations proceed as before. However, by step 7, we see unexpected behavior in the spectrum of $X$. The singular values that began near $0$ suddenly jump up above 1, instead of converging to 1 from below. By step 8, the algorithm is returning complete junk. What happened?

### Spurious Negative Eigenvalues
If you look closely, you can see that the trouble begins in the matrix $R$. By construction, $r_t = x_t^2 \geq 0$, so in exact arithmetic, $R_t$ should be a positive semidefinite matrix.
However, when using `bfloat16`, our plots show that $R$ has negative eigenvalues!
Because $G$ is numerically low rank, $R_0 = GG^\top$ has eigenvalues that are *numerically* equal to zero, and in `bfloat16`, a number like $-10^{-5}$ is numerically equal to zero.
Let's transform the y-axis to emphasize values close to zero and replot this.

In [ ]:
HTML(spectrum_evolution_plot(f16_diagnostics, yscale='asinh', yscale_kw=dict(linear_width=1e-4)).to_jshtml())

Now we see that from the very beginning, $R$ has tiny negative eigenvalues. These eigenvalues represent nothing about the original problem, they are just an artefact of floating point arithemtic. Therefore, we call them "spurious eigenvalues".

These spurious negative eigenvalues start small, but the plot shows that their magnitude grows quickly.
Let's understand mathematically why this happens. Recall the update rule:
$$r_t = r_{t-1} z_t^2 = r_{t-1} h_t(r_{t-1})^2$$
If we now substitute $h_t(x) = \tfrac{15}8 - \tfrac{10}8 x + \tfrac38 x^2$, and plot this update rule, we can see the problem:

In [ ]:
def h(x): return (15/8) - (10/8) * x + (3/8) * x**2
def next_r(r): return r * (h(r)**2)
xxx = np.linspace(-.25, 1, 1000)
fig, ax = plt.subplots()
ax.plot(xxx, next_r(xxx), label='r h(r)^2')
# ax.plot(xxx, xxx, '--', color='gray', linewidth=0.5, label='r')
ax.plot(xxx[:450], ((15/8)**2)*xxx[:450], '--', color='gray', linewidth=0.5, label="(15/8)^2 r")
ax.set_xlabel('r')
ax.legend(loc='lower right')

ax.spines['left'].set_position('zero')
ax.spines['bottom'].set_position('zero')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

As the plot shows, $r_t < \left(\tfrac{15}{8}\right)^2 r_{t-1}$. Thus, if $r_0 < 0$, the magnitude of the spurious eigenvalues grows exponentially! This sets off a chain reaction. As $r_t \to -\infty$, we get $z_t \to \infty$. This causes $q_t \to \infty$ and therefore also $x_t \to \infty$.
This problem cannot be fixed by choosing different polynomials. Conceptually, we are attempting to compute the inverse square root of a negative number. It cannot help but diverge.

To show that the spurious negative eigenvalues of $R_0$ are enough to cause this catestrophic failure, let's rerun the method with every operation in `float64` precision, except that we will convert $R_0$ from `float64` to `bfloat16` and then back to `float64` to induce a little floating point error. As you can see, even this causes a blowup.

In [ ]:
_, posthoc_f16_diagnostics = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=10,
    ambient_dtype=torch.float64,
    xxt_posthoc_dtype=torch.bfloat16,
)(G)

HTML(spectrum_evolution_plot(posthoc_f16_diagnostics).to_jshtml())

## Controlling Negative Eigenvalues by Restarting
If we run Gram Newton Schulz for more than a few iterations, the spurious negative eigenvalues grow unmanageably large. Our solution is simple: run Gram Newton Schulz for only a few iterations.
Rather than using Gram Newton Schulz to compute $X_T$ directly, we use it to compute, say, $X_5$ in a stable manner.
While $X_5$ is not a good approximation to $\lim_{T \to \infty} X_T = \mathrm{polar}(X_0)$, we are closer than when we started.
Now we can apply Gram Newton Schulz a second time on the input $X_5$ to compute $X_{10}$ stably.
We can repeat this over and over to reach whatever $T$ we like.
This restarting technique sacrifices some of the performance gains of Gram Newton Schulz, but it still offers a significant speedup over standard Newton Schulz.

> ### Gram Newton Schulz (Version 2 - Restart Every 5)
> Input: $G \in \R^{m \times n}$ with $m \leq n$ and $\|G\|_2 \leq 1$.
>
> Initialize $X_0 = G$
>
> Repeat for $t = 0, 5, 10, \ldots, T$
> - "Reinitialize" $R_t \gets X_tX_t^\top$ and $Q_t \gets I$.
>
> - Repeat for $s=t+1, \ldots, t+5$:
>   - $Z_s = h_s(R_{s-1})$&emsp;&emsp;(e.g. $\tfrac32 I - \tfrac12 R_{s-1}$)
>   - $R_s = Z_s^\top R_{s-1} Z_s$
>   - $Q_s = Q_{s-1}Z_s$
>
> - $X_{t+5} = Q_{t+5} X_t$
>
> Output: $X_T$

Below we plot this method. As before, we compute $X_{t+s} = Q_{t+s} X_t$ for diagnostic purposes, even though the algorithm does so only for $s=5$. 
As you can see, at iteration $5, 10, 20, 25$, and $30$, $Q$ resets to the identity. Therefore, the eigenvalues of $Q$ never grow beyond $\approx 12$.
Looking closely, you can also see that $R$ develops a some negative eigenvalues, but unlike before, the growth of these eigenvalues is controlled.
Each time we restart, we re-initialize $R = XX^\top$, eliminating any negative eigenvalues of large magnitude.

In [ ]:
_, restart5_diagnostics = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=30,
    restarts=[5, 10, 15, 20, 25],
    ambient_dtype=torch.bfloat16
)(G)

frames = list(range(11)) + [15, 20, 25, 30]
HTML(spectrum_evolution_plot(restart5_diagnostics, frames=frames).to_jshtml())

# When to Restart
Is it safe to run for $r$ iterations without restarting? To avoid numerical trouble, we need to control the magnitude of $Q_r$, even when $R_0$ has spurious negative eigenvalues. The growth of $Q_r$ in turn depends on the specific sequence of polynomials we use.
Furthermore, since the polynomial $p_t$ changes at each iteration, it may not be ideal to restart at regular intervals.
Instead, we can choose when to restart adaptively, depending on the specific sequence of polynomials we have applied since the previous restart.

For the application to Muon, let's now switch over to five iterations of the PolarExpress polynomials.
To obtain a good balance of stability and speed, let's limit ourselves to a single restart.
When should this restart take place?
Using the scalar analogue of the method, let's simulate how the eigenvalues of $Q_t$ evolve when $R_0$ has eigenvalues in the range $[-10^{-4}, 1]$.
The plot below shows that restarting after 2 iterations ensures that $\|Q_t\|_2 \leq 30$ for all iterations. (Restarting after 3 gives a worse bound, but only slightly.)

In [ ]:
SPURIOUS_NEGATIVE = -4  # plants an eigenvalue at -10^-4

spectrum_with_planted_negatives = torch.concat((
    torch.logspace(0, -10, steps=1000, device=DEVICE, dtype=torch.float64),
    -torch.logspace(SPURIOUS_NEGATIVE, -10, steps=100, device=DEVICE, dtype=torch.float64),
))

polar1restart_results = {
    restart: PolarExpressDiagnostic(
        coeffs_name="polar5",
        steps=5,
        restarts=[restart],  # list(range(5)),
        ambient_dtype=torch.bfloat16,
        do_diagnostics=True,
    ).track_eigvals(spectrum_with_planted_negatives)
    for restart in [1, 2, 3, 4]
}

fig, ax = plt.subplots()

for restart, (rs, qs) in polar1restart_results.items():
    ax.plot([np.abs(m).max() for m in qs], label=f"{restart} iter", marker='o')
ax.set_ylabel('Max Eigenvalue of Q_t')
ax.set_xlabel('Iteration')
ax.legend(title="Restart after")
ax.set_yscale('log')
# ax.set_ylim(.9, 100)

our_lim = max(np.abs(vals).max() for vals in polar1restart_results[2][1])
ax.axhline(our_lim, color='black', linestyle='--', linewidth=0.5)
print(f"Max eigenvalue: {our_lim:.2f}")


Now let's run the full method with a restart after the second iteration on our test matrix.

In [ ]:
_, final_diagnostics = PolarExpressDiagnostic(
    coeffs_name="polar5",
    steps=5,
    restarts=[2],
    ambient_dtype=torch.float16,
    do_diagnostics=True,
)(spectrum2matrix(spectrum, aspect_ratio))

HTML(spectrum_evolution_plot(final_diagnostics).to_jshtml())

# Detritus. @Jack you can ignore everything below here

In [ ]:
rs, qs = polar1restart_results[2]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

def update(frame):
    for ax, col, title in zip(axes, (rs, qs), ("R eigvals", "Q eigvals")):
        vals = col[frame].cpu().numpy()
        ax.plot(rs[0].cpu().numpy(), vals, label=f'step {frame}')
        ax.set_title(f'{title} (steps 0 – {frame})')
        ax.set_xlabel('R_0 eigenvalues')
        ax.set_xscale('symlog')
        ax.legend(loc='upper right', fontsize='small')

ani = FuncAnimation(fig, update, frames=len(rs), init_func=lambda: [], interval=500, repeat=False)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
_, restart6_diagnostics = PolarExpressDiagnostic(
    coeffs_name="ns5",
    steps=15,
    # restarts=[5, 10, 15, 20, 25],
    restarts=list(range(0, 15, 6)),
    ambient_dtype=torch.bfloat16
)(G)

frames = None  # list(range(11)) + [15, 20, 25, 30]
HTML(spectrum_evolution_plot(restart6_diagnostics, frames=frames).to_jshtml())

In [ ]:
n = 512
aspect_ratio = 4
spectrum = torch.logspace(-0.01, -8, steps=n, dtype=torch.float64, device=DEVICE)
assert torch.all(spectrum[:-1] >= spectrum[1:]), "spectrum must be decreasing"
# G = torch.diag(spectrum)
G = spectrum2matrix(spectrum, aspect_ratio)

PE = PolarExpressDiagnostic(
    coeffs_name="polar5",
    steps=5,
    restarts=[2],  # list(range(5)),
    ambient_dtype=torch.float64,
    xxt_posthoc_dtype=torch.bfloat16,
    # qx_dtype=torch.bfloat16,
    do_diagnostics=True,
)
out, diagnostics = PE(G)
df = pd.DataFrame(diagnostics)
# since Z and Q are decreasing functions of the corresponding singular value of X_0, flip them for plotting purposes
df['Z_singvals'] = df['Z_singvals'].apply(np.flip)
df['Q_singvals'] = df['Q_singvals'].apply(np.flip)

In [ ]:
restart6_diagnostics.loc[6, 'X_singvals_from_starting_vecs'][:30]

In [ ]:
plt.plot(restart6_diagnostics.loc[0, 'X_singvals_from_starting_vecs'], restart6_diagnostics.loc[6, 'X_singvals_from_starting_vecs'])
plt.xscale('log')
# plt.yscale('log')

In [ ]:
fig, ax = plt.subplots(2, figsize=(5, 4), height_ratios=(4, 1), sharex=True)

ax[0].plot(df['dual_obj'], marker='o', label='dual obj')
ax[0].plot(df['orth_error'], marker='o', label='orth error')
ax[0].plot(df['residual_error'], marker='o', label='residual error')

for restart in PE.restarts:
    ax[0].axvline(x=restart - 0.5, color='black', linestyle='dashed', linewidth=0.5)

ax[0].set_title("Convergence")
ax[0].set_yscale("log")
ax[0].set_xlim(0, len(df) - 1)
ax[0].legend();

ax[1].plot(df['X_max_singval'], marker='o')

ax[1].set_title("Spectral norm of X")
ax[1].set_xlabel("Iteration")

In [ ]:
df.loc[0, ['R_min_singval_from_starting_vecs', 'R_min_singval']]

In [ ]:
import numpy as np
np.allclose(df.loc[0, 'X_singvals_from_starting_vecs'], df.loc[0, 'X_singvals'])

In [ ]:
HTML(spectrum_evolution_plot(df).to_jshtml())

In [ ]:
rs, qs = PE.track_eigvals(torch.linspace(-1e-3, 1, steps=100, device=DEVICE, dtype=torch.float64))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

def update(frame):
    for ax, col, title in zip(axes, (rs, qs), ("R eigvals", "Q eigvals")):
        vals = col[frame].cpu().numpy()
        ax.plot(rs[0].cpu().numpy(), vals, label=f'step {frame}')
        ax.set_title(f'{title} (steps 0 – {frame})')
        ax.set_xlabel('R_0 eigenvalues')
        ax.set_xscale('symlog')
        ax.legend(loc='upper right', fontsize='small')

ani = FuncAnimation(fig, update, frames=len(rs), init_func=lambda: [], interval=500, repeat=False)
plt.close(fig)
HTML(ani.to_jshtml())